# Push Spark-Enriched Data Directly into the Trainer Data Cache

This notebook shows the **write path** proof of concept: `SparkClient` enriches raw reviews and **pushes the result as Arrow record batches straight into the Trainer data cache**, and a `TrainJob` then trains on it. No Iceberg table is read by the cache for the training data.

## What you will learn
- How to start a `TrainJob` whose data cache is empty and waits for data
- How to enrich raw data with `SparkClient` and push it to the cache with Arrow Flight `do_put`
- How the cache head splits the pushed data into contiguous row ranges per cache worker
- How the training workers read their shard from the cache

## Prerequisites
- Kind cluster with Kubeflow Trainer, the Spark Operator and SeaweedFS (S3-compatible) running
- Custom Spark image `spark-s3-iceberg:4.0.4` loaded into the cluster (see `README.md`)
- Data cache image built from the `feat/data-cache-write-path` Trainer branch and set on the `torch-distributed-with-cache` runtime
- Kernel: the SDK virtual environment (`pyarrow`, `pyspark[connect]` installed)
- `kubectl` available on the PATH

## 1. Imports and clients

In [1]:
import os
import re
import socket
import subprocess
import time

import pyarrow as pa
import pyarrow.flight as flight

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import CustomTrainer, DataCacheInitializer, Initializer, TrainerClient

namespace = os.getenv("SPARK_TEST_NAMESPACE", "default")
trainer_client = TrainerClient(backend_config=KubernetesBackendConfig(namespace=namespace))
print(f"TrainerClient ready for namespace: {namespace}")

TrainerClient ready for namespace: default


## 2. Start a Spark Connect session

`get_spark()` (from `spark_seaweed.py` in this folder) creates a Spark Connect session with the S3A and Iceberg settings for SeaweedFS.

In [2]:
from pathlib import Path
import sys

# Make spark_seaweed.py importable wherever the notebook server was started.
for base in (
    Path.cwd(),
    Path.cwd() / "examples/spark/datacache_poc",
    Path.home() / "sdk/examples/spark/datacache_poc",
):
    if (base / "spark_seaweed.py").exists():
        sys.path.insert(0, str(base))
        break

from spark_seaweed import get_spark  # noqa: E402

from kubeflow.spark import SparkClient  # noqa: E402

# Remember existing sessions so cleanup only deletes the one created here.
existing_sessions = {s.name for s in SparkClient().list_sessions()}
spark_client, spark = get_spark()
my_sessions = [s.name for s in spark_client.list_sessions() if s.name not in existing_sessions]
print("Spark version:", spark.version)
print("Session:", my_sessions)

Spark version: 4.0.4
Session: ['spark-connect-09f2d953']


## 3. Create raw reviews

Synthetic, intentionally noisy reviews: HTML tags, nulls, very short text, spam and duplicates.

In [3]:
import random

from pyspark.sql import functions as F  # noqa: N812

random.seed(7)
good = [
    "Great product, works exactly as described and arrived quickly",
    "Terrible quality, it broke after two days of normal use",
    "Average experience, nothing special but does the job well enough",
    "Absolutely love it, would happily buy this again next year",
    "Not worth the money, the battery barely lasts an hour",
]
noise = [None, "ok", "<p>fine</p>", "buy now http://spam.example"]
rows = []
for i in range(400):
    if random.random() < 0.2:
        text = random.choice(noise)
    else:
        text = f"{random.choice(good)} (order {i})"  # unique per row
        if random.random() < 0.3:
            text = f"<b>{text}</b>"
    rows.append((text, random.randint(1, 5)))
rows += random.sample(rows, 40)  # exact duplicates for dropDuplicates to remove

raw = spark.createDataFrame(rows, "content string, rating int")
print("raw rows:", raw.count())

raw rows: 440


## 4. Create a one-row bootstrap table

The cache runtime still expects an Iceberg metadata location when it starts. We give it a tiny placeholder table; the real training data is pushed in step 8 and replaces it.

In [4]:
spark.sql("DROP TABLE IF EXISTS lake.default.cache_bootstrap")
spark.createDataFrame([("bootstrap row", 0)], "content string, label bigint").writeTo(
    "lake.default.cache_bootstrap"
).using("iceberg").create()

METADATA_LOC = spark.sql(
    "SELECT file FROM lake.default.cache_bootstrap.metadata_log_entries ORDER BY timestamp DESC LIMIT 1"
).collect()[0]["file"]
print("Bootstrap metadata:", METADATA_LOC)

Bootstrap metadata: s3a://mlpipeline/warehouse/default/cache_bootstrap/metadata/v1.metadata.json


## 5. Define the training function

The cache is empty of real data when the job starts, so the function first **waits until the pushed rows are visible**. It reads its shard through the cache head, exactly like the pull flow.

In [5]:
def train_func():
    import os
    import time

    import pyarrow.flight as flight
    import torch
    import torch.distributed as dist
    from torch.utils.data import DataLoader, IterableDataset
    from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

    bootstrap_rows = 1  # the placeholder table holds one row

    class CacheDataset(IterableDataset):
        def __init__(self, tokenizer, batch_size=16):
            job = os.getenv("TRAIN_JOB_NAME")
            self.endpoint = f"grpc://{job}-cache-service:50051"
            self.tokenizer = tokenizer
            self.batch_size = batch_size
            self.row_count = 0

        def _descriptor(self):
            rank, world = dist.get_rank(), dist.get_world_size()
            return flight.FlightDescriptor.for_path(str(rank), str(world))

        def wait_for_pushed_data(self, timeout=900):
            client = flight.connect(self.endpoint)
            for _ in range(timeout // 5):
                try:
                    info = client.get_flight_info(self._descriptor())
                    rows = 0
                    for ep in info.endpoints:
                        for loc in ep.locations:
                            reader = flight.FlightClient(loc.uri).do_get(ep.ticket)
                            rows += sum(b.data.num_rows for b in reader)
                    if rows > bootstrap_rows:
                        return
                except Exception as exc:  # cache not ready or data not pushed yet
                    print("waiting for pushed data:", exc)
                time.sleep(5)
            raise TimeoutError("no data was pushed to the cache")

        def __iter__(self):
            self.wait_for_pushed_data()
            client = flight.connect(self.endpoint)
            info = client.get_flight_info(self._descriptor())
            for ep in info.endpoints:
                for loc in ep.locations:
                    reader = flight.FlightClient(loc.uri).do_get(ep.ticket)
                    for rb in reader:
                        batch = rb.data
                        for i in range(0, batch.num_rows, self.batch_size):
                            chunk = batch.slice(i, self.batch_size)
                            self.row_count += chunk.num_rows
                            enc = self.tokenizer(
                                chunk.column("content").to_pylist(),
                                padding="max_length",
                                truncation=True,
                                max_length=64,
                                return_tensors="pt",
                            )
                            yield {
                                "input_ids": enc["input_ids"],
                                "attention_mask": enc["attention_mask"],
                                "labels": torch.tensor(chunk.column("label").to_pylist()),
                            }

    dist.init_process_group(backend="gloo")
    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=5
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    dataset = CacheDataset(tokenizer)
    for batch in DataLoader(dataset, batch_size=None):
        loss = model(
            batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["labels"]
        ).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Loss: {loss.item():.4f}")

    print("Training is finished")
    print(f"Total rows processed: {dataset.row_count}")
    dist.destroy_process_group()

## 6. Submit the TrainJob

`num_data_nodes=2` starts the cache head plus two workers. The pushed rows will be split between them.

In [6]:
job_name = trainer_client.train(
    runtime="torch-distributed-with-cache",
    initializer=Initializer(
        dataset=DataCacheInitializer(
            storage_uri="cache://default/cache_bootstrap",
            metadata_loc=METADATA_LOC,
            iam_role="dummy",
            num_data_nodes=2,
        )
    ),
    trainer=CustomTrainer(
        func=train_func,
        num_nodes=1,
        resources_per_node={"cpu": 1, "memory": "3Gi"},
        packages_to_install=["pyarrow==19.0.0", "transformers==4.45.2"],
    ),
)
print("TrainJob:", job_name)

TrainJob: mc0b2aa13c09


## 7. Wait for the cache and open a port-forward

The push goes to the cache head (`<job>-cache-0`). We forward its Flight port to `localhost:50051`.

In [7]:
def wait_pod_ready(pod, timeout=600):
    end = time.time() + timeout
    while time.time() < end:
        out = subprocess.run(
            [
                "kubectl",
                "get",
                "pod",
                pod,
                "-n",
                namespace,
                "-o",
                "jsonpath={.status.containerStatuses[0].ready}",
            ],
            capture_output=True,
            text=True,
        ).stdout
        if out == "true":
            return
        time.sleep(5)
    raise TimeoutError(f"{pod} not ready")


head_pod = f"{job_name}-cache-0"
# Head plus the two data workers must all be ready before we push.
for pod in (head_pod, f"{head_pod}-1", f"{head_pod}-2"):
    wait_pod_ready(pod)
print("All cache pods are ready")

pf = subprocess.Popen(
    ["kubectl", "port-forward", f"pod/{head_pod}", "50051:50051", "-n", namespace],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
for _ in range(30):
    try:
        socket.create_connection(("localhost", 50051), timeout=1).close()
        break
    except OSError:
        time.sleep(1)
print("Cache head reachable on localhost:50051")

All cache pods are ready
Cache head reachable on localhost:50051


## 8. Enrich with Spark and push to the cache

Cleaning steps: strip HTML, drop null/short/spam rows, drop duplicates, derive `label = rating - 1`, shuffle with a fixed seed. The result is converted to Arrow and sent with `do_put`; the head splits it into one contiguous row range per cache worker.

In [8]:
enriched = (
    raw.withColumn("content", F.regexp_replace("content", "<[^>]+>", ""))
    .filter(F.col("content").isNotNull() & (F.length("content") >= 10))
    .filter(~F.col("content").rlike("(?i)http|buy now"))
    .dropDuplicates(["content"])
    .withColumn("label", (F.col("rating") - 1).cast("long"))
    .filter(F.col("label").between(0, 4))
    .select("content", "label")
    .orderBy(F.rand(7))
)

try:
    table = enriched.toArrow()
except AttributeError:
    table = pa.Table.from_pandas(enriched.toPandas())

schema = pa.schema([("content", pa.string()), ("label", pa.int64())])
table = table.cast(schema)
assert table.num_rows > 0, "nothing to push"

client = flight.connect("grpc://localhost:50051")
options = flight.FlightCallOptions(timeout=120)  # fail instead of hanging
writer, _ = client.do_put(flight.FlightDescriptor.for_path("push"), schema, options)
writer.write_table(table, max_chunksize=500)
writer.close()
pushed_rows = table.num_rows
print(f"Pushed {pushed_rows} enriched rows to the data cache")

Pushed 309 enriched rows to the data cache


## 9. Follow the training logs

The job should report that it processed exactly the number of rows we pushed.

In [9]:
logs = "\n".join(trainer_client.get_job_logs(job_name, follow=True))
print(logs[-1500:])

match = re.findall(r"Total rows processed: (\d+)", logs)
assert match, "training did not report processed rows"
processed = sum(int(m) for m in match)
assert processed == pushed_rows, f"processed {processed} != pushed {pushed_rows}"
print(f"OK: trained on {processed} rows, same as pushed")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Loss: 1.6082
Loss: 1.6033
Loss: 1.6310
Loss: 1.6462
Loss: 1.6487
Loss: 1.6054
Loss: 1.6121
Loss: 1.6336
Loss: 1.6182
Loss: 1.5979
Loss: 1.6007
Loss: 1.5956
Loss: 1.6323
Loss: 1.6068
Loss: 1.6101
Loss: 1.6320
Loss: 1.6348
Loss: 1.6206
Loss: 1.6307
Loss: 1.5957
Training is finished
Total rows processed: 309
OK: trained on 309 rows, same as pushed


## 10. Clean up

In [10]:
pf.terminate()
trainer_client.delete_job(job_name)
spark.sql("DROP TABLE IF EXISTS lake.default.cache_bootstrap")
spark.stop()
for name in my_sessions:
    spark_client.delete_session(name)
print("Cleaned up")

Cleaned up
